# C.4 找得到也選得中嗎？


對「2+2=?」生成三份候選 `[3,3,4]`，正解4已在清單裡。如果選出現最多的答案，最後卻會交3。前置是[候選包含正解的比例](https://birdhackor.github.io/tiny-perceptron-vlm/C.3.html)與[驗證器的範圍](https://birdhackor.github.io/tiny-perceptron-vlm/C.2.html)。本節固定同一份候選，實際比較多數決與算術驗證，讓「找到」和「選中」各自有可觀察的結果。

多數決不理解算術，只數相同答案出現幾次：3有兩次、4有一次，所以3獲選。另一個選擇器用精確規則檢查候選是否等於2+2，第一個3失敗，第二個3也失敗，最後4通過。後者稱verifier，因本例題目本身能用加法得到真值，它有很強的判準；這種優勢不能直接假設存在於未知事實或開放題。

下圖兩條箭頭都從完整候選集合出發。上方數次數而選3，下方查算術而選4，表示平行比較兩種方法；不是先把候選縮成3，再神奇地補出4。左框的coverage命中True，只說集合含正解，無法單獨決定哪條分支最後答對。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/architecture_candidate_selection.svg")))

In [ ]:
from collections import Counter

candidates = [3, 3, 4]
majority = Counter(candidates).most_common(1)[0][0]
verified = next((x for x in candidates if x == 2 + 2), None)
print("集合包含正解", 4 in candidates)
print("多數決", majority, "答對", majority == 4)
print("驗證器", verified, "答對", verified == 4)

`Counter`把答案及次數對應起來，`most_common(1)`取最多次的一項，兩個`[0]`分別取那一筆與其中的答案值。`next`取第一個通過條件的候選；第二參數None代表全部不通過時，回傳「沒有找到」，而非捏造一個候選。輸出依序為True；3、False；4、True，與圖完全一致。

真實評估需要多個問題：先統計各題的候選集合是否含正解，再統計最後選出的答案是否正確。也可以在「確有正解的集合」內，另外統計選中正解的比例，但必須說清楚分母；否則把不同數字都叫selection accuracy會混淆。多數決會在同一誤解大量重複時失敗，驗證器也可能因規則不完整、資料過時或錯讀題目而誤判。

本例能直接算2+2，因此主要目的不是替這道簡單題節省工作，而是核對選擇流程。若改成尚未確認的書店地址，寫一個固定返回「青街8號」的驗證器，只是把答案預埋在程式裡，不能證明一般任務都能選對。選擇能力也需要在獨立題目上測試。

練習只把candidates改成 `[3,3]`，其餘程式不變。先預測集合命中False、多數決仍選3、驗證器返回None且答對False，再執行核對。驗證器無法從沒有正解的清單選出正解；明確保留None，才不會把這種失敗誤記成成功。
